In [ ]:
from huggingface_hub import login, whoami
whoami()


In [ ]:
from huggingface_hub import login, whoami
whoami()

!pip install wandb -Uq

import os, wandb

# ── dedicated workspace for final runs ────────────────────────────────
WANDB_PROJECT = "mdlm-sft-final"            # new project = fresh workspace
WANDB_GROUP   = "stage3-runs_final"           # optional: groups related runs in the UI

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_GROUP"] = WANDB_GROUP

wandb.login()


In [ ]:
import tempfile
from pathlib import Path
from huggingface_hub import list_bucket_tree, download_bucket_files

BUCKET = "avgJo3/final-mdlm-sft-artifacts"

PREFIXES: dict[str, str] = {
    "base":      "20260714T050657-0e2e053501df5170/model_",
    "reasoning": "20260715T045644-7d0f95338c3101b3/model_",
}

CHAIN_STEMS = {
    "train":        "train",
    "mix":          "mix",
    "ablation":     "ablation",
    "mix-ablation": "mix-ablation",
}
ROUNDS = ["R0", "R1", "R2", "R3", "R4"]

_MODEL_TEMPDIRS: dict[str, tempfile.TemporaryDirectory] = {}


def _round_dirname(stem: str, k: int) -> str:
    if stem == "train":
        return f"model_trained_r{k}"
    return f"model_{stem}-trained_r{k}"


def load_model_paths_for(ckpt_alias: str) -> dict[str, dict[str, Path]]:
    if ckpt_alias not in PREFIXES:
        raise KeyError(f"unknown checkpoint alias {ckpt_alias!r}; "
                       f"known aliases: {list(PREFIXES)}")
    prefix = PREFIXES[ckpt_alias]

    files = [
        item for item in list_bucket_tree(BUCKET, prefix=prefix, recursive=True)
        if item.type == "file"
    ]
    if not files:
        raise RuntimeError(f"no files found under prefix: {prefix}")

    tmp = tempfile.TemporaryDirectory(prefix=f"models-{ckpt_alias}-")
    _MODEL_TEMPDIRS[ckpt_alias] = tmp
    tmp_root = Path(tmp.name)

    download_bucket_files(BUCKET, files=[(f, str(tmp_root / f.path)) for f in files])

    data_root = (tmp_root / prefix).parent

    # --- per-chain: R1..R4 only ----------------------------------------------
    def resolve_chain(stem: str) -> dict[str, Path]:
        paths = {}
        for k in range(1, len(ROUNDS)):
            model_dir = data_root / _round_dirname(stem, k)
            assert model_dir.exists(), f"model dir not found: {model_dir}"
            paths[f"R{k}"] = model_dir
        return paths

    result = {cid: resolve_chain(stem) for cid, stem in CHAIN_STEMS.items()}

    # --- R0 shared starting point (both ckpts) -------------------------------
    r0_dir = data_root / "model_trained_r0"
    assert r0_dir.exists(), f"model_trained_r0 not found: {r0_dir}"
    result["model_trained_r0"] = {"model_trained_r0": r0_dir}

    # --- model_base only exists for the base ckpt ----------------------------
    model_base_dir = data_root / "model_base"
    if model_base_dir.exists():
        result["model_base"] = {"model_base": model_base_dir}

    return result


model_paths_by_ckpt: dict[str, dict[str, dict[str, Path]]] = {}
model_paths_by_ckpt["base"]      = load_model_paths_for("base")
model_paths_by_ckpt["reasoning"] = load_model_paths_for("reasoning")

# --- summary -----------------------------------------------------------------
for ckpt_alias, chains in model_paths_by_ckpt.items():
    print(f"\n{ckpt_alias}")
    for chain_id, rounds in chains.items():
        for rk, path in rounds.items():
            print(f"  {chain_id:20s}  {rk:20s}  →  {path.name}")

In [ ]:
from datasets import load_from_disk, load_dataset

dd = load_dataset("avgJo3/gsm8k-processed")
print(dd["main_train"]["completion"])
dd.save_to_disk("gsm8k")

In [ ]:
!git clone https://github.com/AvgJoe-cpu/mdlm_sft.git
%cd mdlm_sft
!pip install -e . -q

In [ ]:

from datetime import datetime, timezone
import secrets
import pprint
import hashlib, time
from datetime import datetime, timezone
from huggingface_hub import sync_bucket, list_bucket_tree
import os, gc, shutil, tempfile, torch
from pathlib import Path
from datasets import load_from_disk, load_dataset

import sys
sys.path.insert(0, "/content/mdlm_sft/src")

from mdlm_sft.mdlm.mdlm_sft_v4 import run_training, MDLMSFTConfig
from mdlm_sft.mdlm.mdlm_gen_v3 import run_inference, MDLMGenerationConfig

In [ ]:
print(run_training)

## EXPERIMENT 3: BASELINE (REASONING VS. BASE) -- NO DEGRADATION

In [ ]:
FINAL_CONFIG = {
    # HPO winners
    "learning_rate":   1.84e-3,     # Stage-B rank-1
    "warmup_ratio":    0.01,         # Stage-A top-15 empirical
    "weight_decay":    0.01,        # Stage-A rank-1
    "time_epsilon":    1e-3,        # ε-sweep midpoint

    # Training procedure pins (identical to sweeps)
    "lr_scheduler_type":           "cosine",
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size":  64,
    "gradient_accumulation_steps": 1,           # B_eff = 320
    "loss_weight_type":            "scheduler",
    "num_train_epochs":  100,

    # Reproducibility
    "seed":       42,
}

TMP_ROOT = "/dev/shm" if os.path.isdir("/dev/shm") else None
SCRATCH  = Path(tempfile.mkdtemp(prefix="mdlm-sft-final-", dir=TMP_ROOT))
print(f"[scratch] {SCRATCH}")

# ── per-run paths ───────────────────────────────────────────────
RUN_PATHS = {
    "M2": {
        "model_name_or_path": "avgJo3/mdlm-sft-M2-100pct-1ep",
        "train_ds_path":      "/content/gsm8k/main_train",
        "eval_ds_path":       "/content/gsm8k/main_validation",
        "gen_input_path":     "/content/gsm8k/main_test",   # must have "prompt" column
    }
}

# ── generation knobs (shared across runs) ───────────────────────
GEN_KW = dict(response_length=128, num_steps=128, batch_size=64)

EVALS_PER_EPOCH = 4
world_size = max(1, torch.cuda.device_count())
eff_batch  = (FINAL_CONFIG["per_device_train_batch_size"]
              * FINAL_CONFIG["gradient_accumulation_steps"]
              * world_size)

try:
    for tag, spec in RUN_PATHS.items():
        paths     = RUN_PATHS[tag]
        run_name  = spec["model_name_or_path"].split("/")[-1]
        model_out = str(SCRATCH / f"model_{run_name}")            # SFT output → gen input
        gen_out   = str(SCRATCH / f"generated_{run_name}")        # gen output

        # ── training cadence ────────────────────────────────────
        n_examples   = len(load_from_disk(paths["train_ds_path"], keep_in_memory=False))
        steps_per_ep = max(1, n_examples // eff_batch)
        cadence      = max(1, steps_per_ep // EVALS_PER_EPOCH)

        print(f"\n=== {tag} → {spec['model_name_or_path']} ===")
        print(f"n={n_examples} | eff_batch={eff_batch} | steps/epoch={steps_per_ep} "
              f"| cadence={cadence} | model_out={model_out}")

        # ── 1. train ────────────────────────────────────────────
        train_cfg = MDLMSFTConfig(
            **FINAL_CONFIG,
            model_name_or_path = paths["model_name_or_path"],
            train_ds_path       = paths["train_ds_path"],
            eval_ds_path        = paths["eval_ds_path"],
            output_dir         = model_out,
            run_name           = run_name,
            logging_strategy   = "steps",
            logging_steps      = cadence,
            eval_strategy      = "steps",
            eval_steps         = cadence,
            save_strategy      = "no",
            eval_on_start      = True,
        )
        run_training(train_cfg)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        # sanity: SFT actually wrote weights
        assert os.path.isdir(model_out) and os.listdir(model_out), \
            f"[FAIL] no model artifacts in {model_out}"

        # ── 2. generate with the freshly-SFT'd model ────────────
        print(f"[gen] {tag}: {paths['gen_input_path']} → {gen_out}")
        gen_cfg = MDLMGenerationConfig(
            model_name_or_path  = model_out,               # <-- reads from SCRATCH
            dataset_input_path  = paths["gen_input_path"],
            dataset_output_path = gen_out,                 # <-- writes into SCRATCH
            **GEN_KW,
        )
        run_inference(gen_cfg)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        assert os.path.isdir(gen_out), f"[FAIL] no generated dataset at {gen_out}"
        print(f"  ✓ {tag}: model + generations done")

    now    = datetime.now(timezone.utc)
    digest = hashlib.sha256(now.isoformat().encode()).hexdigest()[:16]
    run_id = f"gsm8k-{now.strftime('%Y%m%dT%H%M%S')}-{digest}"
    remote = f"hf://buckets/avgJo3/final-mdlm-sft-artifacts/{run_id}"

    delay = 30
    for attempt in range(1, 4):
        try:
            sync_bucket(str(SCRATCH), remote)
            for item in list_bucket_tree("avgJo3/final-mdlm-sft-artifacts", prefix=run_id, recursive=True):
                break
        except Exception as e:
            print(f"[upload] attempt {attempt}/3 failed: {type(e).__name__}: {e}")
            if attempt == 3:
                print(f"[upload] giving up. Local artifacts at: {SCRATCH}")
                raise
            time.sleep(delay)
            delay *= 2
finally:
    shutil.rmtree(SCRATCH, ignore_errors=True)
    print(f"[scratch] cleaned {SCRATCH}")

In [ ]:
import gc, os, time, shutil, hashlib, tempfile
from datetime import datetime, timezone
from pathlib import Path
from datasets import load_from_disk
import torch

from huggingface_hub import list_bucket_tree
# from your training lib:
# from mdlm_sft import MDLMSFTConfig, run_training, MDLMGenerationConfig, run_inference, sync_bucket

FINAL_CONFIG = {
    "learning_rate":   1.84e-3,
    "warmup_ratio":    0.1,
    "weight_decay":    0.01,
    "time_epsilon":    1e-3,
    "lr_scheduler_type":           "cosine",
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size":  64,
    "gradient_accumulation_steps": 5,
    "loss_weight_type":            "scheduler",
    "num_train_epochs":            10,
    "seed":                        42,
}

GEN_KW       = dict(response_length=128, num_steps=128, batch_size=64)
EVALS_PER_EPOCH = 4
EVAL_DS_PATH    = "/content/gsm8k/main_validation"
TRAIN_DS_PATH   = "/content/gsm8k/main_train"
GEN_INPUT_PATH  = "/content/gsm8k/main_test"

TMP_ROOT = "/dev/shm" if os.path.isdir("/dev/shm") else None
SCRATCH  = Path(tempfile.mkdtemp(prefix="mdlm-sft-final-", dir=TMP_ROOT))
print(f"[scratch] {SCRATCH}")

world_size = max(1, torch.cuda.device_count())
eff_batch  = (FINAL_CONFIG["per_device_train_batch_size"]
              * FINAL_CONFIG["gradient_accumulation_steps"]
              * world_size)

try:
    for ckpt_alias, chains in model_paths_by_ckpt.items():
        for chain_id, rounds in chains.items():
            for rk, model_path in rounds.items():

                tag      = f"{ckpt_alias}/{chain_id}/{rk}"
                run_name = f"{ckpt_alias}-{chain_id}-{rk}"
                model_out = str(SCRATCH / f"model_{run_name}")
                gen_out   = str(SCRATCH / f"generated_{run_name}")

                n_examples   = len(load_from_disk(TRAIN_DS_PATH, keep_in_memory=False))
                steps_per_ep = max(1, n_examples // eff_batch)
                cadence      = max(1, steps_per_ep // EVALS_PER_EPOCH)

                print(f"\n=== {tag} ===")
                print(f"  model_path={model_path}")
                print(f"  n={n_examples} | eff_batch={eff_batch} | "
                      f"steps/epoch={steps_per_ep} | cadence={cadence}")

                # ── 1. train ────────────────────────────────────
                train_cfg = MDLMSFTConfig(
                    **FINAL_CONFIG,
                    model_name_or_path = str(model_path),
                    train_ds_path      = TRAIN_DS_PATH,
                    eval_ds_path       = EVAL_DS_PATH,
                    output_dir         = model_out,
                    run_name           = run_name,
                    logging_strategy   = "steps",
                    logging_steps      = cadence,
                    eval_strategy      = "steps",
                    eval_steps         = cadence,
                    save_strategy      = "no",
                    eval_on_start      = True,
                )
                run_training(train_cfg)
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

                assert os.path.isdir(model_out) and os.listdir(model_out), \
                    f"[FAIL] no model artifacts in {model_out}"

                # ── 2. generate ─────────────────────────────────
                print(f"[gen] {tag}: {GEN_INPUT_PATH} → {gen_out}")
                gen_cfg = MDLMGenerationConfig(
                    model_name_or_path  = model_out,
                    dataset_input_path  = GEN_INPUT_PATH,
                    dataset_output_path = gen_out,
                    **GEN_KW,
                )
                run_inference(gen_cfg)
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

                assert os.path.isdir(gen_out), f"[FAIL] no generated dataset at {gen_out}"
                print(f"  ✓ {tag}: done")

    # ── upload ──────────────────────────────────────────────────
    now    = datetime.now(timezone.utc)
    digest = hashlib.sha256(now.isoformat().encode()).hexdigest()[:16]
    run_id = f"gsm8k-{now.strftime('%Y%m%dT%H%M%S')}-{digest}"
    remote = f"hf://buckets/avgJo3/final-mdlm-sft-artifacts/{run_id}"

    delay = 30
    for attempt in range(1, 4):
        try:
            sync_bucket(str(SCRATCH), remote)
            for item in list_bucket_tree("avgJo3/final-mdlm-sft-artifacts",
                                         prefix=run_id, recursive=True):
                break
            print(f"[upload] ✓ run_id={run_id}")
            break
        except Exception as e:
            print(f"[upload] attempt {attempt}/3 failed: {type(e).__name__}: {e}")
            if attempt == 3:
                print(f"[upload] giving up. Local artifacts at: {SCRATCH}")
                raise
            time.sleep(delay)
            delay *= 2

finally:
    shutil.rmtree(SCRATCH, ignore_errors=True)
    print(f"[scratch] cleaned {SCRATCH}")

In [ ]:
import gc, os, time, shutil, hashlib, tempfile
from datetime import datetime, timezone
from pathlib import Path
from datasets import load_from_disk
import torch

from huggingface_hub import list_bucket_tree
# from your training lib:
# from mdlm_sft import MDLMSFTConfig, run_training, MDLMGenerationConfig, run_inference, sync_bucket

FINAL_CONFIG = {
    "learning_rate":   1.84e-3,
    "warmup_ratio":    0.1,
    "weight_decay":    0.01,
    "time_epsilon":    1e-3,
    "lr_scheduler_type":           "cosine",
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size":  64,
    "gradient_accumulation_steps": 5,
    "loss_weight_type":            "scheduler",
    "num_train_epochs":            10,
    "seed":                        42,
}

GEN_KW       = dict(response_length=128, num_steps=128, batch_size=64)
EVALS_PER_EPOCH = 4
EVAL_DS_PATH    = "/content/gsm8k/main_validation"
TRAIN_DS_PATH   = "/content/gsm8k/main_train"
GEN_INPUT_PATH  = "/content/gsm8k/main_test"

# ── single model to run ─────────────────────────────────────────
MODEL_PATH = "avgJo3/mdlm-sft-M2-100pct-1ep"        # <-- set this
RUN_NAME   = "single-run"                # <-- used for output naming/tagging

TMP_ROOT = "/dev/shm" if os.path.isdir("/dev/shm") else None
SCRATCH  = Path(tempfile.mkdtemp(prefix="mdlm-sft-final-", dir=TMP_ROOT))
print(f"[scratch] {SCRATCH}")

world_size = max(1, torch.cuda.device_count())
eff_batch  = (FINAL_CONFIG["per_device_train_batch_size"]
              * FINAL_CONFIG["gradient_accumulation_steps"]
              * world_size)

try:
    tag       = RUN_NAME
    run_name  = RUN_NAME
    model_out = str(SCRATCH / f"model_{run_name}")
    gen_out   = str(SCRATCH / f"generated_{run_name}")

    n_examples   = len(load_from_disk(TRAIN_DS_PATH, keep_in_memory=False))
    steps_per_ep = max(1, n_examples // eff_batch)
    cadence      = max(1, steps_per_ep // EVALS_PER_EPOCH)

    print(f"\n=== {tag} ===")
    print(f"  model_path={MODEL_PATH}")
    print(f"  n={n_examples} | eff_batch={eff_batch} | "
          f"steps/epoch={steps_per_ep} | cadence={cadence}")

    # ── 1. train ────────────────────────────────────
    train_cfg = MDLMSFTConfig(
        **FINAL_CONFIG,
        model_name_or_path = str(MODEL_PATH),
        train_ds_path      = TRAIN_DS_PATH,
        eval_ds_path       = EVAL_DS_PATH,
        output_dir         = model_out,
        run_name           = run_name,
        logging_strategy   = "steps",
        logging_steps      = cadence,
        eval_strategy      = "steps",
        eval_steps         = cadence,
        save_strategy      = "no",
        eval_on_start      = True,
    )
    run_training(train_cfg)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    assert os.path.isdir(model_out) and os.listdir(model_out), \
        f"[FAIL] no model artifacts in {model_out}"

    # ── 2. generate ─────────────────────────────────
    print(f"[gen] {tag}: {GEN_INPUT_PATH} → {gen_out}")
    gen_cfg = MDLMGenerationConfig(
        model_name_or_path  = model_out,
        dataset_input_path  = GEN_INPUT_PATH,
        dataset_output_path = gen_out,
        **GEN_KW,
    )
    run_inference(gen_cfg)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    assert os.path.isdir(gen_out), f"[FAIL] no generated dataset at {gen_out}"
    print(f"  ✓ {tag}: done")

    # ── upload ──────────────────────────────────────────────────
    now    = datetime.now(timezone.utc)
    digest = hashlib.sha256(now.isoformat().encode()).hexdigest()[:16]
    run_id = f"gsm8k-{now.strftime('%Y%m%dT%H%M%S')}-{digest}"
    remote = f"hf://buckets/avgJo3/final-mdlm-sft-artifacts/{run_id}"

    delay = 30
    for attempt in range(1, 4):
        try:
            sync_bucket(str(SCRATCH), remote)
            for item in list_bucket_tree("avgJo3/final-mdlm-sft-artifacts",
                                         prefix=run_id, recursive=True):
                break
            print(f"[upload] ✓ run_id={run_id}")
            break
        except Exception as e:
            print(f"[upload] attempt {attempt}/3 failed: {type(e).__name__}: {e}")
            if attempt == 3:
                print(f"[upload] giving up. Local artifacts at: {SCRATCH}")
                raise
            time.sleep(delay)
            delay *= 2

finally:
    shutil.rmtree(SCRATCH, ignore_errors=True)
    print(f"[scratch] cleaned {SCRATCH}")

In [ ]:
from huggingface_hub import list_bucket_tree, download_bucket_files
from datasets import load_from_disk
from pathlib import Path
import tempfile

BUCKET   = "avgJo3/final-mdlm-sft-artifacts"
RUN_ID   = "gsm8k-20260715T191953-aee6f56c2aaf1851"2
GEN_DIR  = "generated_mdlm-sft-M2-100pct-1ep"
PREFIX   = f"{RUN_ID}/{GEN_DIR}"

tmp = tempfile.TemporaryDirectory(prefix="mdlm-sft-download-")
tmp_root = Path(tmp.name)

files = [
    item for item in list_bucket_tree(BUCKET, prefix=PREFIX, recursive=True)
    if item.type == "file"
]
print(f"[download] found {len(files)} file(s) under {PREFIX}")

download_bucket_files(BUCKET, files=[(f, str(tmp_root / f.path)) for f in files])

ds = load_from_disk(str(tmp_root / PREFIX))

ref_ds = load_from_disk("/content/gsm8k/main_test")


In [ ]:
print(generated_ds["completion"][10])

In [ ]:
import re

_LAST_NUM_RE = re.compile(r"(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*$")

def extract_answer(example):
    # strip trailing whitespace/punctuation then grab last number
    text = example["completion"].strip().rstrip(".")
    match = _LAST_NUM_RE.search(text)
    return {"answer_raw": match.group(1).replace(",", "") if match else None}

generated_ds = generated_ds.map(extract_answer)
ds = generated_ds.filter(lambda example: example["answer_raw"] is not None)
print(f"before: {len(generated_ds)} | after: {len(ds)} | dropped: {len(generated_ds) - len(ds)}")

ref_ds = load_from_disk("/content/gsm8k/main_test")
print(ref_ds)

In [ ]:
ref_ds = load_from_disk("/content/gsm8k/main_test")
print(ref_ds["completion"])

In [ ]:
import re

_LAST_NUM_RE = re.compile(r"(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*$")

def extract_answer(example):
    text = example["completion"].strip().rstrip(".")
    match = _LAST_NUM_RE.search(text)
    return {"answer_raw": match.group(1).replace(",", "") if match else None}

generated_ds = generated_ds.map(extract_answer)
ds = generated_ds.filter(lambda example: example["answer_raw"] is not None)
print(f"before: {len(generated_ds)} | after: {len(ds)} | dropped: {len(generated_ds) - len(ds)}")

ref_ds = load_from_disk("/content/gsm8k/main_test")

# --- sort both by id ---------------------------------------------------------
ds      = ds.sort("id")
ref_ds  = ref_ds.sort("id")



In [ ]:
from datasets import Dataset, concatenate_datasets
from jinja2 import Template
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

from huggingface_hub import list_bucket_tree, download_bucket_files
from datasets import load_from_disk
from pathlib import Path
import tempfile


In [ ]:

BUCKET   = "avgJo3/final-mdlm-sft-artifacts"
RUN_ID   = "gsm8k-20260715T191953-aee6f56c2aaf1851"
GEN_DIR  = "generated_mdlm-sft-M2-100pct-1ep"
PREFIX   = f"{RUN_ID}/{GEN_DIR}"

tmp = tempfile.TemporaryDirectory(prefix="mdlm-sft-download-")
tmp_root = Path(tmp.name)

files = [
    item for item in list_bucket_tree(BUCKET, prefix=PREFIX, recursive=True)
    if item.type == "file"
]
print(f"[download] found {len(files)} file(s) under {PREFIX}")

download_bucket_files(BUCKET, files=[(f, str(tmp_root / f.path)) for f in files])

ds = load_from_disk(str(tmp_root / PREFIX))

ref_ds = load_from_disk("/content/gsm8k/main_test")



In [ ]:
import json
import re

_JSON_RE = re.compile(r"```json\s*(\{.*?\})\s*```", re.DOTALL)

def parse_fn(batch: dict) -> dict:
    results = {
        "trace_to_answer":    [],
        "trace_to_trace":     [],
        "trace_to_prose":     [],
        "trace_to_reference": [],
        "reasoning":          [],
        "parse_ok":           [],
    }

    for raw in batch["judge_raw"]:
        match = _JSON_RE.search(raw)
        try:
            parsed = json.loads(match.group(1)) if match else json.loads(raw)
            results["trace_to_answer"].append(parsed.get("trace_to_answer"))
            results["trace_to_trace"].append(parsed.get("trace_to_trace"))
            results["trace_to_prose"].append(parsed.get("trace_to_prose"))
            results["trace_to_reference"].append(parsed.get("trace_to_reference"))
            results["reasoning"].append(parsed.get("reasoning"))
            results["parse_ok"].append(True)
        except (json.JSONDecodeError, AttributeError):
            results["trace_to_answer"].append(None)
            results["trace_to_trace"].append(None)
            results["trace_to_prose"].append(None)
            results["trace_to_reference"].append(None)
            results["reasoning"].append(None)
            results["parse_ok"].append(False)

    return results

ds = ds.map(parse_fn, batched=True)
print(f"parse failures: {ds['parse_ok'].count(False)}/{len(ds)}")

In [ ]:
import numpy as np

SCORE_COLS = ["trace_to_answer", "trace_to_trace", "trace_to_prose", "trace_to_reference"]

for col in SCORE_COLS:
    values = [v for v in ds[col] if v is not None]
    print(f"{col:25s}  mean={np.mean(values):+.2f}  n={len(values)}")

In [ ]:
from huggingface_hub import list_bucket_tree, download_bucket_files
from pathlib import Path



def download_datasets_1(
    bucket,
    run,
    data_dir,
) -> None:

    def dataset_files(remote_name: str) -> list[tuple[str, str]]:
        return [
            (f"{run}/{remote_name}/data-00000-of-00001.arrow", f"{data_dir}/{remote_name}/data-00000-of-00001.arrow"),
            (f"{run}/{remote_name}/dataset_info.json",         f"{data_dir}/{remote_name}/dataset_info.json"),
            (f"{run}/{remote_name}/state.json",                f"{data_dir}/{remote_name}/state.json"),
        ]

    ds_files = [
        # train chain
        *dataset_files("generated_base-ablation-R1"),
        *dataset_files("generated_base-ablation-R2"),
        *dataset_files("generated_base-ablation-R3"),
        *dataset_files("generated_base-ablation-R4"),

        # mix chain
        *dataset_files("generated_base-mix-R1"),
        *dataset_files("generated_base-mix-R2"),
        *dataset_files("generated_base-mix-R3"),
        *dataset_files("generated_base-mix-R4"),

        # ablation chain
        *dataset_files("generated_base-mix-ablation-R1"),
        *dataset_files("generated_base-mix-ablation-R2"),
        *dataset_files("generated_base-mix-ablation-R3"),
        *dataset_files("generated_base-mix-ablation-R4"),

        *dataset_files("generated_base-model_trained_r0-model_trained_r0"),
        *dataset_files("generated_base-train-R1"),
        *dataset_files("generated_base-train-R2"),
        *dataset_files("generated_base-train-R3"),
        *dataset_files("generated_base-train-R4"),

        *dataset_files("generated_reasoning-ablation-R1"),
        *dataset_files("generated_reasoning-ablation-R2"),
        *dataset_files("generated_reasoning-ablation-R3"),
        *dataset_files("generated_reasoning-ablation-R4"),

        *dataset_files("generated_reasoning-mix-R1"),
        *dataset_files("generated_reasoning-mix-R2"),
        *dataset_files("generated_reasoning-mix-R3"),
        *dataset_files("generated_reasoning-mix-R4"),

        *dataset_files("generated_reasoning-mix-ablation-R1"),
        *dataset_files("generated_reasoning-mix-ablation-R2"),
        *dataset_files("generated_reasoning-mix-ablation-R3"),
        *dataset_files("generated_reasoning-mix-ablation-R4"),

        *dataset_files("generated_reasoning-model_trained_r0-model_trained_r0"),
        *dataset_files("generated_reasoning-train-R1"),
        *dataset_files("generated_reasoning-train-R2"),
        *dataset_files("generated_reasoning-train-R3"),
        *dataset_files("generated_reasoning-train-R4"),
    ]

    print(f"files to download: {len(ds_files)}")
    download_bucket_files(bucket_id=bucket, files=ds_files)
    print("done.")


def download_datasets_2(
    bucket,
    run,
    data_dir,
) -> None:

    def dataset_files(remote_name: str) -> list[tuple[str, str]]:
        return [
            (f"{run}/{remote_name}/data-00000-of-00001.arrow", f"{data_dir}/{remote_name}/data-00000-of-00001.arrow"),
            (f"{run}/{remote_name}/dataset_info.json",         f"{data_dir}/{remote_name}/dataset_info.json"),
            (f"{run}/{remote_name}/state.json",                f"{data_dir}/{remote_name}/state.json"),
        ]

    ds_files = [
        *dataset_files("generated_single-run"),
    ]

    print(f"files to download: {len(ds_files)}")
    download_bucket_files(bucket_id=bucket, files=ds_files)
    print("done.")


download_datasets_1(bucket="avgJo3/final-mdlm-sft-artifacts", run="gsm8k-20260717T232848-5278902f403b190c", data_dir="./data")
download_datasets_2(bucket="avgJo3/final-mdlm-sft-artifacts", run="gsm8k-20260718T005150-2121adce3c40f13d", data_dir="./data")

In [ ]:
from datasets import load_from_disk, DatasetDict
from pathlib import Path

def build_dataset_dict(data_dir: str) -> DatasetDict:
    data_dir = Path(data_dir)

    dataset_names = sorted(
        p.name for p in data_dir.iterdir()
        if p.is_dir() and (p / "dataset_info.json").exists() and (p / "state.json").exists()
    )

    print(f"found {len(dataset_names)} datasets to load")

    datasets = {}
    for name in dataset_names:
        print(f"loading {name}...")
        datasets[name] = load_from_disk(str(data_dir / name))

    return DatasetDict(datasets)


dataset_dict = build_dataset_dict(data_dir="./data")
dataset_dict

In [ ]:
import re


_LAST_NUM_RE = re.compile(r"(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*$")

def extract_answer(example):
    text = example["completion"].strip().rstrip(".")
    match = _LAST_NUM_RE.search(text)
    return {"answer_raw": match.group(1).replace(",", "") if match else None}


original_lengths = {}
new_datasets = {}
for name, ds in dataset_dict.items():
    generated_ds = ds.map(extract_answer)
    filtered_ds = generated_ds.filter(lambda example: example["answer_raw"] is not None)
    original_lengths[name] = len(generated_ds)
    print(f"{name}: before: {len(generated_ds)} | after: {len(filtered_ds)} | dropped: {len(generated_ds) - len(filtered_ds)}")
    new_datasets[name] = filtered_ds

dataset_dict = DatasetDict(new_datasets)

In [ ]:
ref_answers = {ex["id"]: ex["answer_raw"] for ex in ref_ds.map(extract_answer)}

def compute_accuracy(example):
    ref = ref_answers.get(example["id"])
    return {"correct": ref is not None and example["answer_raw"] == ref}

results = {}
for name, ds in dataset_dict.items():
    original_len = original_lengths[name]  # size before the answer_raw filter, see note below
    scored = ds.map(compute_accuracy)
    n_correct = sum(scored["correct"])
    n_subset = len(scored)

    subset_acc = n_correct / n_subset if n_subset else 0.0
    normalized_acc = n_correct / original_len if original_len else 0.0

    results[name] = {
        "correct": n_correct,
        "subset_size": n_subset,
        "original_size": original_len,
        "subset_accuracy": subset_acc,
        "normalized_accuracy": normalized_acc,
    }
    print(f"{name}: {n_correct}/{n_subset} (subset) = {subset_acc:.4f} | "
          f"{n_correct}/{original_len} (normalized) = {normalized_acc:.4f}")

In [ ]:
total_correct = sum(r["correct"] for r in results.values())
total_original = sum(r["original_size"] for r in results.values())
total_subset = sum(r["subset_size"] for r in results.values())

naive_mean_acc = sum(r["subset_accuracy"] for r in results.values()) / len(results)
weighted_acc = total_correct / total_original  # true pooled accuracy, size-weighted by construction

print(f"naive mean of per-dataset accuracies: {naive_mean_acc:.4f}  <-- misleading, ignores subset sizes")
print(f"weighted (pooled) accuracy: {total_correct}/{total_original} = {weighted_acc:.4f}")

In [ ]:
weighted_acc

In [ ]:
ref_size = len(ref_ds)

ref_answers = {ex["id"]: ex["answer_raw"] for ex in ref_ds.map(extract_answer)}

def compute_accuracy(example):
    ref = ref_answers.get(example["id"])
    return {"correct": ref is not None and example["answer_raw"] == ref}

results = {}
for name, ds in dataset_dict.items():
    original_len = original_lengths[name]  # size before the answer_raw filter
    scored = ds.map(compute_accuracy)
    n_correct = sum(scored["correct"])
    n_subset = len(scored)

    subset_acc = n_correct / n_subset if n_subset else 0.0
    normalized_acc = n_correct / original_len if original_len else 0.0
    ref_acc = n_correct / ref_size if ref_size else 0.0

    results[name] = {
        "correct": n_correct,
        "subset_size": n_subset,
        "original_size": original_len,
        "ref_size": ref_size,
        "subset_accuracy": subset_acc,
        "normalized_accuracy": normalized_acc,
        "ref_accuracy": ref_acc,
    }
    print(f"{name}: {n_correct}/{n_subset} (subset) = {subset_acc:.4f} | "
          f"{n_correct}/{original_len} (normalized) = {normalized_acc:.4f} | "
          f"{n_correct}/{ref_size} (of ref) = {ref_acc:.4f}")

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(results, orient="index")
df.index.name = "dataset"
df = df.reset_index()

df = df[[
    "dataset", "correct", "subset_size", "original_size", "ref_size",
    "subset_accuracy", "normalized_accuracy", "ref_accuracy",
]]

df = df.sort_values("dataset").reset_index(drop=True)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
df

In [ ]:
df = pd.DataFrame.from_dict(results, orient="index")
df.index.name = "dataset"
df = df.reset_index()

df = df[[
    "dataset", "correct", "subset_size", "ref_size",
    "subset_accuracy", "ref_accuracy",
]].rename(columns={"subset_size": "size", "subset_accuracy": "accuracy"})

df = df.sort_values("dataset").reset_index(drop=True)
df

In [ ]:
latex_str = df.to_latex(
    index=False,
    float_format="%.4f",
    caption="GSM8K accuracy by generated dataset",
    label="tab:gsm8k_accuracy",
    column_format="l" + "r" * (len(df.columns) - 1),
    escape=True,
)

with open(f"accuracy_results.tex", "w") as f:
    f.write(latex_str)

print(latex_str)

In [ ]:
from datasets import load_dataset
ref_ds = load_dataset("avgJo3/gsm8k-processed", split='main_test')

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"

model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()

JUDGE_TEMPLATE = Template("""You are an expert evaluator of reasoning traces generated by language models.

## Task
Evaluate the reasoning trace generated in response to a math problem according to the criteria below.
Return your evaluation as a JSON object — **no prose, no markdown fences, just the raw JSON**.

## Criteria

### (1) Internal Relevance  (scale: -100 to 100)
- **(a) trace_to_answer**: Is the reasoning trace relevant to and supportive of the final answer?
- **(b) trace_to_trace**: If multiple reasoning traces are present, are they mutually consistent and relevant to each other?
- **(c) trace_to_prose**: Is the reasoning trace relevant to the surrounding prose / explanatory text?

### (2) Reference Relevance  (scale: -100 to 100)
- **(d) trace_to_reference**: How well does the generated reasoning trace align with the reference thinking trace?

Negative scores indicate the element is actively misleading or contradictory.
Zero indicates complete irrelevance.
Positive scores indicate relevance and alignment.

## Problem
{{ question }}

## Generated Response
{{ answer }}

## Reference Response
{{ gold_completion }}


## Output format (strict)
```json
{
  "trace_to_answer":    <int, -100 to 100>,
  "trace_to_trace":     <int, -100 to 100>,
  "trace_to_prose":     <int, -100 to 100>,
  "trace_to_reference": <int, -100 to 100>,
  "reasoning":          "<one sentence justification>"
}
```""")



def inference_fn(batch: dict) -> dict:

    prompts = [
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": JUDGE_TEMPLATE.render(
                    question=q,
                    answer=a,
                    gold_completion=g,
                )},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for q, a, g in zip(batch["prompt"], batch["completion"], batch["gold_completion"])
    ]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,          # judge prompt is longer than generation prompt
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    completions = tokenizer.batch_decode(
        outputs[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    return {"judge_raw": completions}



In [ ]:
from datasets import concatenate_datasets

merged_datasets = {}
for name, ds in dataset_dict.items():
    ds = ds.sort("id")
    ref_ds_sorted = ref_ds.sort("id").rename_columns({"completion": "gold_completion"})

    cols_to_keep = ["gold_completion"]
    ref_ds_sorted = ref_ds_sorted.select_columns(cols_to_keep)

    merged_ds = concatenate_datasets([ds, ref_ds_sorted], axis=1)
    print(f"{name}: {len(ds)} rows merged with gold_completion")
    merged_datasets[name] = merged_ds

dataset_dict = DatasetDict(merged_datasets)

In [ ]:
import json
import re

_JSON_RE = re.compile(r"```json\s*(\{.*?\})\s*```", re.DOTALL)

def parse_fn(batch: dict) -> dict:
    results = {
        "trace_to_answer":    [],
        "trace_to_trace":     [],
        "trace_to_prose":     [],
        "trace_to_reference": [],
        "reasoning":          [],
        "parse_ok":           [],
    }

    for raw in batch["judge_raw"]:
        match = _JSON_RE.search(raw)
        try:
            parsed = json.loads(match.group(1)) if match else json.loads(raw)
            results["trace_to_answer"].append(parsed.get("trace_to_answer"))
            results["trace_to_trace"].append(parsed.get("trace_to_trace"))
            results["trace_to_prose"].append(parsed.get("trace_to_prose"))
            results["trace_to_reference"].append(parsed.get("trace_to_reference"))
            results["reasoning"].append(parsed.get("reasoning"))
            results["parse_ok"].append(True)
        except (json.JSONDecodeError, AttributeError):
            results["trace_to_answer"].append(None)
            results["trace_to_trace"].append(None)
            results["trace_to_prose"].append(None)
            results["trace_to_reference"].append(None)
            results["reasoning"].append(None)
            results["parse_ok"].append(False)

    return results


dataset_dict = dataset_dict.map(inference_fn, batched=True, batch_size=64)
#dataset_dict = dataset_dict.map(parse_fn, batched=True)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from jinja2 import Template
import torch

In [ ]:
import json
import re

_JSON_RE = re.compile(r"```json\s*(\{.*?\})\s*```", re.DOTALL)

def parse_fn(batch: dict) -> dict:
    results = {
        "trace_to_answer":    [],
        "trace_to_trace":     [],
        "trace_to_prose":     [],
        "trace_to_reference": [],
        "reasoning":          [],
        "parse_ok":           [],
    }

    for raw in batch["judge_raw"]:
        match = _JSON_RE.search(raw)
        try:
            parsed = json.loads(match.group(1)) if match else json.loads(raw)
            results["trace_to_answer"].append(parsed.get("trace_to_answer"))
            results["trace_to_trace"].append(parsed.get("trace_to_trace"))
            results["trace_to_prose"].append(parsed.get("trace_to_prose"))
            results["trace_to_reference"].append(parsed.get("trace_to_reference"))
            results["reasoning"].append(parsed.get("reasoning"))
            results["parse_ok"].append(True)
        except (json.JSONDecodeError, AttributeError):
            results["trace_to_answer"].append(None)
            results["trace_to_trace"].append(None)
            results["trace_to_prose"].append(None)
            results["trace_to_reference"].append(None)
            results["reasoning"].append(None)
            results["parse_ok"].append(False)

    return results

dataset_dict = dataset_dict.map(parse_fn, batched=True)


In [ ]:
dataset_dict

In [ ]:
import datasets
import numpy as np

def compute_metric_means(dataset):
    """Compute the mean of the 4 trace metric columns for a single dataset."""
    cols = ['trace_to_answer', 'trace_to_trace', 'trace_to_prose', 'trace_to_reference']
    means = {col: np.mean(dataset[col]) for col in cols}
    means['average'] = np.mean(list(means.values()))
    return means

# Apply to all datasets in the DatasetDict
results = {
    name: compute_metric_means(ds)
    for name, ds in dataset_dict.items()
}

# Pretty print
for name, means in results.items():
    print(f"\n--- {name} ---")
    for col, val in means.items():
        print(f"  {col}: {val:.4f}")

In [ ]:
import numpy as np
import pandas as pd

def compute_metric_means(dataset):
    """Compute mean and normalized mean of the 4 trace metric columns."""
    cols = ['trace_to_answer', 'trace_to_trace', 'trace_to_prose', 'trace_to_reference']

    means = {col: np.mean(dataset[col]) for col in cols}
    means['average'] = np.mean(list(means.values()))

    # Normalize from [-100, 100] → [0, 1]
    normalized = {f"{col}_norm": (val + 100) / 200 for col, val in means.items()}

    return {**means, **normalized}

# Apply to all datasets
results = {
    name: compute_metric_means(ds)
    for name, ds in dataset_dict.items()
}

# DataFrame for easy comparison
df = pd.DataFrame(results).T
print(df.round(4))

In [ ]:
rename_map = {
    'trace_to_answer':        r'Ans',
    'trace_to_trace':         r'Trace',
    'trace_to_prose':         r'Prose',
    'trace_to_reference':     r'Ref',
    'average':                r'Avg',
    'trace_to_answer_norm':   r'Ans$_n$',
    'trace_to_trace_norm':    r'Trace$_n$',
    'trace_to_prose_norm':    r'Prose$_n$',
    'trace_to_reference_norm':r'Ref$_n$',
    'average_norm':           r'Avg$_n$',
}

latex_str = (
    df.round(4)
    .rename(columns=rename_map)
    .to_latex(
        caption="Trace metric means (raw and normalized)",
        label="tab:trace_metrics",
        position="h",
        bold_rows=True,
        column_format="l" + "r" * len(df.columns)
    )
)

with open("trace_metrics.tex", "w") as f:
    f.write(latex_str)

In [ ]:
rename_map = {
    'trace_to_answer_norm':    r'Ans',
    'trace_to_trace_norm':     r'Trace',
    'trace_to_prose_norm':     r'Prose',
    'trace_to_reference_norm': r'Ref',
    'average_norm':            r'Avg',
}

norm_cols = list(rename_map.keys())

latex_str = (
    df[norm_cols]
    .round(4)
    .rename(columns=rename_map)
    .to_latex(
        caption="Trace metric means (normalized)",
        label="tab:trace_metrics",
        position="h",
        bold_rows=True,
        column_format="l" + "r" * len(norm_cols)
    )
)

with open("trace_metrics.tex", "w") as f:
    f.write(latex_str)